In [ ]:
import duckdb
import pandas as pd
import statsmodels.api as sm
import statsmodels.formula.api as smf


In [ ]:
import numpy as np

In [ ]:
# define 'admission with sepsis' by combination logic: infection and acute organ dysfunction  
sepsis_cohort=duckdb.sql("""
WITH explicit_sepsis AS (
    -- Method 1: Explicit Sepsis Codes anywhere in the sequence
    SELECT DISTINCT subject_id, hadm_id
    FROM 'diagnoses_icd.csv'
    WHERE (icd_version = 10 AND (icd_code LIKE 'A40%' OR icd_code LIKE 'A41%' OR icd_code LIKE 'R652%'))
       OR (icd_version = 9 AND (icd_code IN ('99591', '99592', '78552') OR icd_code LIKE '038%'))
),

infection_claims AS (
    -- Method 2a: Infection component of combination logic
    SELECT DISTINCT subject_id, hadm_id
    FROM 'diagnoses_icd.csv'
    WHERE 
        -- Bacterial Pneumonia, UTI, Intra-abdominal, Skin/Soft Tissue, Bacteremia
        (icd_version = 10 AND (icd_code LIKE 'J18%' OR icd_code LIKE 'N390%' OR icd_code LIKE 'K65%' OR icd_code LIKE 'A41%'))
     OR (icd_version = 9 AND (icd_code LIKE '486%' OR icd_code LIKE '5990%' OR icd_code LIKE '567%' OR icd_code LIKE '038%'))
),

organ_dysfunction AS (
    -- Method 2b: Organ dysfunction component of combination logic
    SELECT DISTINCT subject_id, hadm_id
    FROM 'diagnoses_icd.csv'
     WHERE 
        -- Acute Renal Failure, Respiratory Failure, Septic Shock, Lactic Acidosis
        (icd_version = 10 AND (icd_code LIKE 'N17%' OR icd_code LIKE 'J960%' OR icd_code LIKE 'R652%' OR icd_code = 'E872'))
     OR (icd_version = 9 AND (icd_code LIKE '584%' OR icd_code LIKE '51881%' OR icd_code LIKE '78552%' OR icd_code = '2762'))
),

implicit_sepsis_combination AS (
    -- Intersection: Must have BOTH an infection AND organ dysfunction
    SELECT inf.subject_id, inf.hadm_id
    FROM infection_claims inf
    JOIN organ_dysfunction org 
        ON inf.hadm_id = org.hadm_id
)

SELECT subject_id, hadm_id FROM explicit_sepsis
UNION
SELECT subject_id, hadm_id FROM implicit_sepsis_combination
""").df()

print(sepsis_cohort)

In [ ]:
# calculate readmittime
admission_30d=duckdb.sql("""
WITH NextAdmissions AS (
    SELECT 
        subject_id,
        hadm_id,
        admittime,
        dischtime,
        -- Fetch the admittime of the patient's next hospital stay
        LEAD(admittime) OVER (
            PARTITION BY subject_id 
            ORDER BY admittime
        ) AS next_admittime
    FROM 'admissions.csv'
)
SELECT 
    subject_id,
    hadm_id,
    admittime,
    dischtime,
    next_admittime,
    -- Calculate days between current discharge and next admission
    DATE_DIFF('day', dischtime, next_admittime) AS days_to_readmit,
    -- Binary flag for 30-day readmission (1 = Readmitted within 30 days, 0 = No)
    CASE 
        WHEN DATE_DIFF('day', dischtime, next_admittime) BETWEEN 0 AND 30 THEN 1 
        ELSE 0 
    END AS readmitted_30d
FROM NextAdmissions
;""").df()

print(admission_30d)



In [ ]:
#join sepsis cohort with 30 day readmission 
sepsis_readm=duckdb.sql("""
SELECT subject_id,hadm_id, days_to_readmit,readmitted_30d,
    CASE 
        WHEN hadm_id IN (SELECT hadm_id FROM sepsis_cohort)THEN 1
        ELSE 0
        END AS sepsis_cohort
FROM admission_30d
;""").df()

print(sepsis_readm)

In [ ]:
sepsis_readm.to_csv("sepsis_readm.csv")